In [ ]:
import torch
import torch.nn.functional as F
import requests
import math
from tqdm.auto import tqdm
import random

/Users/xerneas/projects/makemore/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()

print(words[:10])
len(words)


['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


32033

In [3]:
itos = [chr(i + ord('a')) for i in range(26)]
itos.append('')  # 26 = end token

def encode(s):
    return [ord(c) - ord('a') for c in s]

def decode(indices):
    return ''.join(itos[i] for i in indices if i != 26)

enc = []
for w in words:
    enc.append(encode(w))

print(enc[:10])

[[4, 12, 12, 0], [14, 11, 8, 21, 8, 0], [0, 21, 0], [8, 18, 0, 1, 4, 11, 11, 0], [18, 14, 15, 7, 8, 0], [2, 7, 0, 17, 11, 14, 19, 19, 4], [12, 8, 0], [0, 12, 4, 11, 8, 0], [7, 0, 17, 15, 4, 17], [4, 21, 4, 11, 24, 13]]


In [8]:
#weights
W = torch.randn((27, 27), requires_grad=True)
lrs = 5
xs, ys = [], []
for i in range(len(enc)):
    xs.append(torch.tensor([26] + enc[i], dtype=torch.long))
    ys.append(torch.tensor(enc[i] + [26], dtype=torch.long))


In [ ]:
for j in range(10):
    order = list(range(len(enc)))
    random.shuffle(order)

    pbar = tqdm(order, desc=f"epoch {j+1}/10")
    for k, i in enumerate(pbar):
        y = ys[i]
        hot = F.one_hot(xs[i], num_classes=27).float()
        #forward
        logits = hot @ W
        cnt = logits.exp()
        P = cnt/cnt.sum(1, keepdim=True)

        # print(P)
        # print(P.sum(1))

        #backward
        loss = -P[torch.arange(len(P)), y].log().mean() + 0.01 * (W**2).mean()
        loss.backward()
        # print(loss)
        if k % 500 == 0 or k == len(order) - 1:
            pbar.set_postfix(loss=float(loss.item()))
        
        #update
        W.data -= (lrs * math.e**(-j/8)) * W.grad
        W.grad.zero_()
    




epoch 10/10: 100%|██████████| 32033/32033 [00:02<00:00, 14232.08it/s, loss=1.99]


In [1]:
for i in range(100):
    out = []
    hot = F.one_hot(torch.tensor([26]), num_classes=27).float()
    logits = hot @ W
    cnt = logits.exp()
    P = cnt/cnt.sum(1, keepdim=True)
    ix = torch.multinomial(P[0], num_samples=1).item()
    out.append(ix)
    while ix != 26:
        hot = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = hot @ W
        cnt = logits.exp()
        P = cnt/cnt.sum(1, keepdim=True)
        ix = torch.multinomial(P[0], num_samples=1).item()
        out.append(ix)
    print(out, decode(out))
    

NameError: name 'F' is not defined